In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import cdsapi

# Example list of lat/lon points
points = [
    (52.5535, -118.0147)
]

c = cdsapi.Client()



# Separate lats and lons
lats = [p[0] for p in points]
lons = [p[1] for p in points]

# Determine bounding box
north = max(lats) + 1.0   # add small buffer
south = min(lats) - 1.0
east  = max(lons) + 1.0
west  = min(lons) - 1.0

# Define download parameters
c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': ['10m_u_component_of_wind', '10m_v_component_of_wind'],
        'year': '2024',
        'month': '01',
        'day': '02',
        'time': [
            '07:00',
        ],
        'area': [north, west, south, east],  # N, W, S, E
        'format': 'netcdf'
    },
    'era5_region.nc'
)



2025-10-04 17:15:17,401 INFO Request ID is b38e20e9-2fa0-4dd6-b98a-c8c945d72de3
2025-10-04 17:15:19,515 INFO status has been updated to accepted
2025-10-04 17:15:31,944 INFO status has been updated to successful


24bf958e31514e6e860b4507c70ffa13.nc:   0%|          | 0.00/33.3k [00:00<?, ?B/s]

'era5_region.nc'

In [ ]:

# Open ERA5 NetCDF file (already downloaded)
ds = xr.open_dataset("era5_region.nc")

# Separate lats and lons into arrays for vectorized interpolation
lats = xr.DataArray([p[0] for p in points], dims="points")
lons = xr.DataArray([p[1] for p in points], dims="points")

# Interpolate u10 and v10 components to all points at once
u_interp = ds['u10'].interp(latitude=lats, longitude=lons)
v_interp = ds['v10'].interp(latitude=lats, longitude=lons)

# Convert to wind speed and direction
wind_speed = np.sqrt(u_interp**2 + v_interp**2)
wind_dir = (np.degrees(np.arctan2(u_interp, v_interp)) + 360) % 360


print(wind_speed)
print(wind_dir)

<xarray.DataArray (valid_time: 1, points: 1)> Size: 8B
array([[1.56807288]])
Coordinates:
    number      int64 8B 0
  * valid_time  (valid_time) datetime64[ns] 8B 2024-01-02T07:00:00
    expver      <U4 16B '0001'
    latitude    (points) float64 8B 52.55
    longitude   (points) float64 8B -118.0
Dimensions without coordinates: points
<xarray.DataArray 'u10' (valid_time: 1, points: 1)> Size: 8B
array([[354.60516065]])
Coordinates:
    number      int64 8B 0
  * valid_time  (valid_time) datetime64[ns] 8B 2024-01-02T07:00:00
    expver      <U4 16B '0001'
    latitude    (points) float64 8B 52.55
    longitude   (points) float64 8B -118.0
Dimensions without coordinates: points
